In [1]:
import sys
sys.path.append("/Users/sujaladhikari/Desktop/FedIDS")

In [2]:
import os 
import numpy as np 
import pandas as pd 
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import torch 
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from FederatedLearningModel.fedmodel import MLP
from torch.optim import Adam

In [3]:
dataset = pd.read_csv('../silos_datasets/combined_binary_silos.csv')
dataset = dataset.drop(columns = 'Unnamed: 0')
dataset.head(5)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label_Binary
0,60453,94,1,1,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,0
1,53904,63,1,1,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,0
2,80,83165164,7,6,354,11595,348,0,50.571429,131.172732,...,20,51.0,0.0,51,51,83100000.0,0.0,83100000,83100000,1
3,80,85013328,6,6,361,11595,361,0,60.166667,147.377633,...,32,1014.0,0.0,1014,1014,84900000.0,0.0,84900000,84900000,1
4,80,83303368,8,5,337,11595,337,0,42.125000,119.147493,...,32,1011.0,0.0,1011,1011,83200000.0,0.0,83200000,83200000,1


In [4]:
print(dataset['Label_Binary'].value_counts())
print(f"The dataset has following number {dataset.shape[0]} number of rows and {dataset.shape[1]} number of columns")

Label_Binary
0    556488
1    556488
Name: count, dtype: int64
The dataset has following number 1112976 number of rows and 79 number of columns


In [5]:
random_seed = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
X = dataset.drop(columns = 'Label_Binary')
X = X.to_numpy()
y = dataset['Label_Binary']
y = y.to_numpy()


In [7]:
## Spliting the data into train, test, and validation splits
X_train , X_vals, y_train, y_vals = train_test_split(X,y , test_size=0.3, random_state=random_seed)
X_val, X_test, y_val, y_test = train_test_split(X_vals, y_vals, test_size=0.5, random_state=random_seed)

In [8]:
### Standard Scaling for all the datas

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_train = torch.tensor(X_train, dtype = torch.float32)
y_train = torch.tensor(y_train, dtype = torch.long)

X_val = scaler.transform(X_val)
X_val = torch.tensor(X_val, dtype = torch.float32)
y_val = torch.tensor(y_val, dtype = torch.long)

X_test = scaler.transform(X_test)
X_test = torch.tensor(X_test, dtype = torch.float32)
y_test = torch.tensor(y_test, dtype = torch.long)



In [9]:
training_data = TensorDataset(X_train,y_train)
testing_data = TensorDataset(X_test, y_test)
validation_data = TensorDataset(X_val, y_val)


In [10]:
training_batch = DataLoader(training_data, batch_size = 64, shuffle = True)
testing_batch = DataLoader(training_data, batch_size=64, shuffle=False)
validation_batch = DataLoader(validation_data,batch_size = 64, shuffle = False)

In [11]:
print(MLP)

<class 'FederatedLearningModel.fedmodel.MLP'>


In [ ]:
input_size = dataset.shape[0]
hidden_layer = [256, 128,64,8]
num_classes = 2

In [13]:
## We will be using the same model that is being used in fedmodel.py for the federated purpose
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(input_size, hidden_layer, num_classes).to(device)
### Optimizer and loss function 
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)
loss = nn.CrossEntropyLoss()
num_epoch = 10
model = model.to(device)

In [ ]:
### Training, testing and evaluation of the centralized model 
## setting up the training environment:
def train(model, train_loader,validation_loader, criterion, device):
    model.train()
    training_loss = 0.0
    train_correct = 0
    training_total = 0 
    for samples, classes in train_loader:
        samples = samples.to(device)
        classes = classes.to(device)
        prediciton = model(samples)
        optimizer.zero_grad()
        loss = criterion(prediciton, classes)
        loss.backward()
        optimizer.step()
        training_loss =loss.item() * samples.size(0) ## Calculating the loss for each batch 
        _,predicted = prediciton.max(1)
        training_total += classes.size(0)
        train_correct += predicted.eq(classes).sum().item() ## Converting the total predicted that are equal to the classes and sum them 
    training_loss += training_loss/len(train_loader)
    training_accuracy = 100 * train_correct/ training_total 
    val_loss = 0
    val_correct = 0
    val_total = 0 
    model.eval()
    with torch.no_grad():
        for val_batch in validation_loader:
            features, labels = val_batch 
            features = features.to(device)
            labels = labels.to(device)
            val_outputs = model(features)
            val_loss_batch = criterion(val_outputs, labels)
            val_loss += val_loss_batch.item() * features.size(0)
            val_correct += val_outputs.eq(labels).sum().item()
            val_total += labels.size(0)
    avg_validation_loss = val_loss / len(validation_loader.dataset)
    validation_accuracy = 100 * val_correct/ val_total
    val_loss.apend(avg_validation_loss)
    return training_loss, training_accuracy , val_loss, validation_accuracy, val_correct, train_correct